加载工具

In [4]:
from dotenv import load_dotenv
import os

# 关键：每次运行都强制覆盖系统环境变量
load_dotenv(override=True)

# 然后读取
api_key = os.getenv('DEEPSEEK_API_KEY')

工具

In [5]:
from langchain_tavily import TavilySearch

web_search = TavilySearch(
    max_results=5,
    topic="general",
)

添加记忆管理m

In [12]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# 连接sqilte
conn = sqlite3.connect('resources/personal_chief.db',check_same_thread=False)
# 初始化checkpointer
checkpointer = SqliteSaver(conn)
# 自动建表
checkpointer.setup()


初始化模型

In [13]:
from langchain.chat_models import init_chat_model
import os

model = init_chat_model(
    model="qwen3.5-plus",  # 模型名称，这里选择qwen3.5-plus，这是一个多模态模型
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

定义智能体m

In [14]:
from langchain.agents import create_agent

system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
"""

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer,
)

测试m

In [15]:
from langchain.messages import HumanMessage

# 准备多模态消息，图片是网络上的冰箱食物图片
multimodal_message = HumanMessage(
    content=[
        {"type": "image",
         "url": "https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg"},
        {"type": "text", "text": "帮我看看这些食材能做些什么？"}
    ])

config = {"configurable": {"thread_id": "6"}}

response = agent.invoke({"messages": [multimodal_message]}, config)

In [16]:
# 友好打印
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_197fede647de46c58cf264a4)
 Call ID: call_197fede647de46c58cf264a4
  Args:
    query: 三文鱼 鸡胸肉 西兰花 蘑菇 彩椒 番茄 健康食谱
    search_depth: advanced
================================= Tool Message =================================
Name: tavily_search

{"query": "三文鱼 鸡胸肉 西兰花 蘑菇 彩椒 番茄 健康食谱", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://cookpad.com/tw/%E9%A3%9F%E8%AD%9C/13394466", "title": "三文魚&雞胸肉食譜與作法by Suky   - Cookpad", "content": "1. 煎飯三明治：剩飯料理\n2. 蔥香椒鹽蝦\n3. 清爽雞肉春筍\n4. 【厚生廚房】黑糖檸檬愛玉\n\nCookpad 在 Cookpad App 開啟\n\nCookpad Premium 訂閱會員 [...] 已收藏    \n\n收藏食譜以便日後查看\n\n## 料理步驟\n\n25mins\n\n1. 1\n\

继续询问

In [17]:
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢第1道菜，可以说的更详细点吗？")]},
    config
)

# 友好打印
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

# 🍽️ 烤三文鱼鸡胸肉配时蔬 - 详细烹饪指南

非常棒的选择！这道菜营养均衡、操作简单，而且颜值超高。下面是为您量身定制的详细做法：

---

## 📋 食材准备（2-3人份）

| 类别 | 食材 | 用量 | 备注 |
|------|------|------|------|
| **主料** | 三文鱼 | 2块（约300g） | 您的冰箱里有 |
| | 鸡胸肉 | 1块（约200g） | 您的冰箱里有 |
| **蔬菜** | 西兰花 | 半颗 | 切小朵 |
| | 彩椒 | 红/黄各半个 | 切块 |
| | 小番茄 | 8-10颗 | 对半切或整颗 |
| | 蘑菇 | 5-6朵 | 对半切 |
| | 白萝卜 | 半根（可选） | 切片增加清甜 |
| **调味** | 橄榄油 | 2汤匙 | 可用普通食用油代替 |
| | 盐 | 适量 | 约1茶匙 |
| | 黑胡椒 | 适量 | 现磨更香 |
| | 柠檬 | 半个 | 切片装饰用 |
| | 大蒜 | 2瓣（可选） | 切片增香 |

---

## 🔪 详细步骤

### **第1步：食材预处理（10分钟）**

1. **处理肉类**
   - 三文鱼和鸡胸肉用清水冲洗干净，用厨房纸吸干表面水分
   - 鸡胸肉如果太厚，可从中间横切一刀使其厚度均匀（便于熟透）
   - 两面均匀撒上盐和黑胡椒，轻轻按摩入味
   - **腌制时间**：至少15分钟（理想30分钟）

2. **处理蔬菜**
   - 西兰花切成一口大小的小朵
   - 彩椒去籽后切成2-3厘米的块状
   - 小番茄洗净，可对半切或保持整颗
   - 蘑菇洗净对半切
   - 白萝卜去皮切成薄片（约0.5cm厚）
   - 柠檬切成薄片备用

---

### **第2步：摆盘准备（5分钟）**

1. **烤盘处理**
   - 在烤盘上铺一层锡纸或烘焙纸（方便清理）
   - 刷一层薄薄的橄榄油防粘

2. **摆放顺序**（关键！）
   ```
   底层：根茎类蔬菜（白萝卜片、土豆如果有）
   中层：西兰花、彩椒、蘑菇
   上层：三文

3.2开发智能体